# BirdCLEF+ 2026 - exp001 Submission

**CPU Notebook / 推論専用 / 90分以内**

### Kaggle Notebook の Input に追加するもの
- `birdclef-2026` … コンペデータ
- `birdclef2026-exp001-weights` … Colabで学習した重み (best_fold0.pth)

In [ ]:
!pip install -q timm

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import timm
from tqdm.notebook import tqdm

# GPU提出は1分制限のためCPU固定
DEVICE = torch.device('cpu')
print(f'torch     : {torch.__version__}')
print(f'torchaudio: {torchaudio.__version__}')
print(f'device    : {DEVICE}')

In [ ]:
import glob

# ── パス自動検索（Kaggle Dataset名・パスが変わっても対応）────
# 重みファイル
_w = glob.glob('/kaggle/input/**/best_fold0.pth', recursive=True)
WEIGHT_PATH = _w[0] if _w else '/kaggle/input/birdclef2026-exp001-weights/best_fold0.pth'

# コンペデータ（sample_submission.csv の場所から逆引き）
_s = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
COMP_DIR = os.path.dirname(_s[0]) if _s else '/kaggle/input/birdclef-2026'

SAMPLE_SUB_CSV = f'{COMP_DIR}/sample_submission.csv'
TEST_SOUND_DIR = f'{COMP_DIR}/test_soundscapes'

print(f'COMP_DIR     : {COMP_DIR}')
print(f'WEIGHT_PATH  : {WEIGHT_PATH}')
print(f'weight exists: {os.path.exists(WEIGHT_PATH)}')

In [ ]:
# ── ハイパーパラメータ（train時と同一にすること）─────────────
CFG = dict(
    sample_rate      = 32000,
    n_samples        = 32000 * 5,
    n_mels           = 128,
    n_fft            = 1024,
    hop_length       = 320,
    fmin             = 20,
    fmax             = 16000,
    model_name       = 'tf_efficientnet_b0_ns',
    num_classes      = 234,
    in_channels      = 1,
    infer_batch_size = 32,
)

In [ ]:
# ── torchaudio による高速Mel変換 ──────────────────────────────
mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate = CFG['sample_rate'],
        n_fft       = CFG['n_fft'],
        hop_length  = CFG['hop_length'],
        n_mels      = CFG['n_mels'],
        f_min       = CFG['fmin'],
        f_max       = CFG['fmax'],
    ),
    T.AmplitudeToDB(top_db=80),
)

def audio_to_melspec(audio_tensor: torch.Tensor) -> torch.Tensor:
    """(batch, n_samples) -> (batch, 1, n_mels, time)"""
    with torch.no_grad():
        mel = mel_transform(audio_tensor)
    mel = mel - mel.amin(dim=(-2, -1), keepdim=True)
    mel = mel / (mel.amax(dim=(-2, -1), keepdim=True) + 1e-8)
    return mel.unsqueeze(1)

In [ ]:
# ── モデル定義（train時と同一）────────────────────────────────
class BirdCLEFModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'], pretrained=False,
            in_chans=CFG['in_channels'], num_classes=0, global_pool='avg',
        )
        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(self.backbone.num_features, CFG['num_classes']),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

In [ ]:
# ── モデルロード ──────────────────────────────────────────────
checkpoint = torch.load(WEIGHT_PATH, map_location='cpu')
model = BirdCLEFModel()
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
LABELS = checkpoint['labels']
print(f'Loaded: epoch={checkpoint["epoch"]}, CV AUC={checkpoint["best_auc"]:.4f}')
print(f'Classes: {len(LABELS)}')

In [ ]:
# ── チャンク切り出し ──────────────────────────────────────────
def extract_chunks(audio: torch.Tensor, end_secs: list) -> torch.Tensor:
    """1つのsoundscapeから複数チャンクをまとめて切り出す。"""
    n, sr = CFG['n_samples'], CFG['sample_rate']
    chunks = []
    for end_sec in end_secs:
        end   = end_sec * sr
        start = end - n
        if start < 0:
            chunk = torch.cat([torch.zeros(-start), audio[:end]])
        elif end > len(audio):
            chunk = torch.cat([audio[start:], torch.zeros(n - len(audio[start:]))])
        else:
            chunk = audio[start:end]
        chunks.append(chunk)
    return torch.stack(chunks)  # (N, n_samples)

In [ ]:
# ── 推論メインループ ──────────────────────────────────────────
sub_df = pd.read_csv(SAMPLE_SUB_CSV)

def parse_row_id(row_id):
    parts = row_id.rsplit('_', 1)
    return parts[0], int(parts[1])

sub_df[['stem', 'end_sec']] = sub_df['row_id'].apply(
    lambda x: pd.Series(parse_row_id(x))
)
all_preds = np.zeros((len(sub_df), len(LABELS)), dtype=np.float32)
bs = CFG['infer_batch_size']

for stem, group in tqdm(sub_df.groupby('stem'), desc='Soundscapes'):
    ogg_path = os.path.join(TEST_SOUND_DIR, f'{stem}.ogg')
    if not os.path.exists(ogg_path):
        continue

    waveform, sr = torchaudio.load(ogg_path)
    if sr != CFG['sample_rate']:
        waveform = torchaudio.functional.resample(waveform, sr, CFG['sample_rate'])
    audio = waveform.mean(dim=0)  # モノラル化

    indices  = group.index.tolist()
    end_secs = group['end_sec'].tolist()
    chunks   = extract_chunks(audio, end_secs)   # (N, n_samples)

    for i in range(0, len(chunks), bs):
        specs = audio_to_melspec(chunks[i:i + bs])   # (B, 1, n_mels, time)
        with torch.no_grad():
            preds = torch.sigmoid(model(specs)).numpy()
        all_preds[indices[i:i + bs]] = preds

print('Inference done.')

In [ ]:
# ── submission.csv の保存 ──────────────────────────────────────
result_df = pd.DataFrame(all_preds, columns=LABELS)
result_df.insert(0, 'row_id', sub_df['row_id'].values)
result_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved: shape={result_df.shape}')
result_df.head(3)